In [13]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [11]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('dezc') \
    .getOrCreate()

spark.version

26/03/09 01:33:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


'4.1.1'

In [12]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet', header=True, inferSchema=True)
df = df.repartition(4)
df.write.parquet('yellow', 'overwrite')

In [17]:
df.filter(F.date_trunc('day', df.tpep_pickup_datetime)=='2025-11-15') \
    .count()

162604

In [44]:
df.createOrReplaceTempView('yellow')
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [57]:
spark.sql("""
select
    (unix_timestamp(tpep_dropoff_datetime)-unix_timestamp(tpep_pickup_datetime))/3600 as duration
from
    yellow
order by 1 desc
limit 1
""").show()

[Stage 118:>                                                        (0 + 2) / 2]

+-----------------+
|         duration|
+-----------------+
|90.64666666666666|
+-----------------+



In [42]:
zones = spark.read.csv('taxi_zone_lookup.csv', header=True, inferSchema=True)

In [24]:
zones.schema

StructType([StructField('LocationID', IntegerType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [26]:
df_joined = df.join(zones, on=zones.LocationID == df.PULocationID, how='left')

In [41]:
df_joined.groupby('Zone').count().sort('count').head()

Row(Zone="Governor's Island/Ellis Island/Liberty Island", count=1)

In [32]:
df_joined.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio